# Viettel AI Race — Self-hosted inference dưới 9B

Notebook này được thiết kế để **Run All** trên Kaggle:

1. tự dò source hoặc khôi phục source nhúng, rồi dò 100 file input, Qwen3-8B, multilingual-e5-small và hai knowledge base;
2. kiểm tra GPU, dependency và ngân sách tham số;
3. chạy smoke test;
4. chạy inference đủ 100 bản ghi hoàn toàn local;
5. validate schema/offset/cấu trúc ZIP;
6. tạo liên kết tải `/kaggle/working/output.zip`.

Trước khi chạy, chỉ cần attach Dataset có thư mục `input/`; package `medical_coder` đã có fallback nhúng trong notebook. Nếu chưa attach weights hoặc terminology, notebook có thể tải Qwen3-8B, E5-small, ICD-10-CM FY2026 từ CDC và RxNorm Current Prescribable Content 07/2026 từ NLM. Với Tesla P100, notebook tự thay wheel CUDA 12.8 không tương thích bằng PyTorch CUDA 12.6 có `sm_60`. Sau provision, model được đọc bằng `local_files_only=True`; notebook không dùng OpenAI API hay Hugging Face Inference API.

## 1. Cấu hình

Để `None` để notebook tự dò dưới `/kaggle/input`. Nếu tự dò nhầm hoặc có nhiều Dataset tương tự, điền đường dẫn tuyệt đối tương ứng.

In [ ]:
from pathlib import Path

# Source fallback được đóng gói ngay trong notebook để Dataset chỉ cần chứa input.
EMBEDDED_SOURCE_B64 = "..."

# Dataset private đã upload. Nếu Kaggle mount theo alias ngắn, cell tự dò sẽ fallback.
PROJECT_ROOT_OVERRIDE = "/kaggle/input/datasets/thanhhiepvo/viettelairace"
INPUT_DIR_OVERRIDE = "/kaggle/input/datasets/thanhhiepvo/viettelairace/input"
QWEN_MODEL_DIR_OVERRIDE = None
E5_MODEL_DIR_OVERRIDE = None
ICD_KB_PATH_OVERRIDE = None
RXNORM_KB_PATH_OVERRIDE = None

# Nếu chưa attach model weights, tải snapshot về /kaggle/working bằng HF token trong Add-ons.
DOWNLOAD_MODELS_IF_MISSING = True
HF_SECRET_CANDIDATES = ("HF_TOKEN", "HF_KEY", "HUGGINGFACE_TOKEN", "HUGGINGFACE_KEY", "HUGGING_FACE_KEY", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACEHUB_API_TOKEN")
QWEN_REPO_ID = "Qwen/Qwen3-8B"
E5_REPO_ID = "intfloat/multilingual-e5-small"

# Tự tạo terminology từ nguồn chính thức nếu Dataset chưa có hai file TSV.
AUTO_PROVISION_TERMINOLOGY = True
ICD_ARCHIVE_URL = "https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Publications/ICD10CM/2026/icd10cm-Code%20Descriptions-2026.zip"
ICD_ARCHIVE_MD5 = "90c6ec7b89e81d6bc8017f3cf566393d"
RXNORM_ARCHIVE_URL = "https://download.nlm.nih.gov/rxnorm/RxNorm_full_prescribe_07062026.zip"
RXNORM_ARCHIVE_MD5 = "767678e3b5b1d6fe358b61c21659f3ef"

# Cấu hình inference.
INSTALL_DEPENDENCIES = True
RUN_SMOKE_TEST = True
SMOKE_IDS = "1,2"
RESET_OUTPUT_BEFORE_RUN = True
QUANTIZATION = "4bit"            # Khuyến nghị cho GPU Kaggle 16 GB.
USE_SEMANTIC_RETRIEVAL = True
EMBEDDING_DEVICE = "cuda"        # E5 nhỏ; GPU giúp xây index nhanh hơn, chỉ nạp một bản weights.
MAX_CANDIDATES = 3
RETRIEVAL_TOP_K = 20
MAX_INPUT_TOKENS = 24576
MAX_NEW_TOKENS = 4096              # JSON stopping criteria thường dừng sớm hơn nhiều.

WORKING_DIR = Path("/kaggle/working")
MODEL_DOWNLOAD_DIR = WORKING_DIR / "models"
TERMINOLOGY_DIR = WORKING_DIR / "terminology"
OUTPUT_DIR = WORKING_DIR / "output"
CACHE_DIR = WORKING_DIR / "medical_coder_cache"
ZIP_PATH = WORKING_DIR / "output.zip"
LOG_PATH = WORKING_DIR / "inference.log"
RUN_REPORT_PATH = WORKING_DIR / "run_report.json"

assert QUANTIZATION in {"4bit", "8bit", "none"}
assert EMBEDDING_DEVICE in {"cpu", "cuda"}
assert RETRIEVAL_TOP_K >= MAX_CANDIDATES >= 1
print("Configuration loaded.")

## 2. Tự dò các Dataset đã attach

Cell ưu tiên tài nguyên đã attach. Nếu Dataset chỉ có `input/`, nó khôi phục package nhúng vào `/kaggle/working/embedded_viettel_ai_race`. Nếu thiếu, nó tải và chuyển đổi terminology chính thức vào `/kaggle/working/terminology`; checksum và số lượng mã được kiểm tra trước khi inference.

In [ ]:
import base64
import csv
import hashlib
import io
import json
import os
import re
import subprocess
import sys
import urllib.request
import zipfile
from collections import defaultdict
from typing import Callable, Iterable

KAGGLE_INPUT = Path("/kaggle/input")
if not KAGGLE_INPUT.is_dir():
    raise RuntimeError("Notebook này phải chạy trên Kaggle: không thấy /kaggle/input")

def _override(value):
    if not value:
        return None
    path = Path(value).expanduser().resolve()
    if not path.exists():
        print(f"Override chưa tồn tại, chuyển sang tự dò: {path}")
        return None
    return path

def _ranked_choice(candidates: Iterable[Path], label: str, score: Callable[[Path], int]) -> Path:
    unique = sorted({path.resolve() for path in candidates if path.exists()})
    if not unique:
        raise FileNotFoundError(
            f"Không tìm thấy {label}. Hãy attach Dataset tương ứng hoặc điền biến *_OVERRIDE."
        )
    ranked = sorted(unique, key=lambda path: (-score(path), len(str(path)), str(path)))
    selected = ranked[0]
    print(f"{label}: {selected}")
    if len(ranked) > 1 and score(ranked[0]) == score(ranked[1]):
        print(f"  Cảnh báo: có {len(ranked)} lựa chọn; đang dùng lựa chọn xếp hạng cao nhất.")
    return selected

def _valid_project_root(path):
    return bool(
        path
        and (path / "src" / "medical_coder" / "__init__.py").is_file()
        and (path / "config" / "model_manifest.json").is_file()
    )

def _materialize_embedded_project():
    root = WORKING_DIR / "embedded_viettel_ai_race"
    marker = root / "src" / "medical_coder" / "__init__.py"
    stamp = root / ".embedded_source_sha256"
    payload = base64.b64decode(EMBEDDED_SOURCE_B64, validate=True)
    payload_sha256 = hashlib.sha256(payload).hexdigest()
    installed_sha256 = stamp.read_text(encoding="utf-8").strip() if stamp.is_file() else ""
    if not marker.is_file() or installed_sha256 != payload_sha256:
        print("Đang cài/cập nhật source nhúng từ notebook...")
        root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(io.BytesIO(payload)) as archive:
            unsafe = [name for name in archive.namelist() if Path(name).is_absolute() or ".." in Path(name).parts]
            if unsafe:
                raise RuntimeError(f"Embedded source chứa path không an toàn: {unsafe[:3]}")
            archive.extractall(root)
        stamp.write_text(payload_sha256 + "\n", encoding="utf-8")
    if not _valid_project_root(root):
        raise RuntimeError("Không thể khôi phục package medical_coder từ notebook.")
    print(f"Source project (embedded): {root}")
    return root

project_override = _override(PROJECT_ROOT_OVERRIDE)
if project_override and not _valid_project_root(project_override):
    print(f"Override chỉ chứa dữ liệu, không chứa source medical_coder: {project_override}")
    project_override = None

project_candidates = [
    path.parent
    for path in list(KAGGLE_INPUT.rglob("pyproject.toml")) + list(WORKING_DIR.rglob("pyproject.toml"))
    if _valid_project_root(path.parent)
]
if project_override:
    PROJECT_ROOT = project_override
    print(f"Source project: {PROJECT_ROOT}")
elif project_candidates:
    PROJECT_ROOT = _ranked_choice(
        project_candidates,
        "Source project",
        lambda path: 10 if (path / "config" / "model_manifest.json").is_file() else 0,
    )
else:
    PROJECT_ROOT = _materialize_embedded_project()

input_override = _override(INPUT_DIR_OVERRIDE)
if input_override:
    INPUT_DIR = input_override
else:
    input_candidates = []
    for one_file in KAGGLE_INPUT.rglob("1.txt"):
        parent = one_file.parent
        if all((parent / f"{index}.txt").is_file() for index in range(1, 101)):
            input_candidates.append(parent)
    INPUT_DIR = _ranked_choice(
        input_candidates,
        "Input directory",
        lambda path: 10 if path.name.casefold() == "input" else 0,
    )

def _load_config(path: Path) -> dict:
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}

MODEL_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

def _all_model_configs():
    return list(KAGGLE_INPUT.rglob("config.json")) + list(MODEL_DOWNLOAD_DIR.rglob("config.json"))

def _qwen_candidates(config_paths):
    result = []
    for config_path in config_paths:
        config = _load_config(config_path)
        if config.get("model_type") == "qwen3" and int(config.get("num_hidden_layers", 0)) >= 30:
            result.append(config_path.parent)
    return result

def _e5_candidates(config_paths):
    return [
        path.parent
        for path in config_paths
        if "multilingual-e5-small" in str(path.parent).casefold()
        or "multilingual_e5_small" in str(path.parent).casefold()
    ]

def _read_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError:
        return None, None
    client = UserSecretsClient()
    for secret_name in HF_SECRET_CANDIDATES:
        try:
            token = client.get_secret(secret_name)
        except Exception:
            continue
        if token:
            return token, secret_name
    return None, None

def _ensure_huggingface_hub():
    try:
        from huggingface_hub import snapshot_download
        return snapshot_download
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "huggingface-hub>=0.30,<2"],
            check=True,
        )
        from huggingface_hub import snapshot_download
        return snapshot_download

config_paths = _all_model_configs()
initial_qwen_candidates = _qwen_candidates(config_paths)
initial_e5_candidates = _e5_candidates(config_paths)
if DOWNLOAD_MODELS_IF_MISSING and (not initial_qwen_candidates or not initial_e5_candidates):
    snapshot_download = _ensure_huggingface_hub()
    hf_token, hf_secret_name = _read_hf_token()
    if hf_secret_name:
        print(f"Đã đọc Hugging Face token từ Kaggle Secret: {hf_secret_name} (không hiển thị giá trị)")
    else:
        print("Không tìm thấy HF secret theo các tên cấu hình; thử tải public snapshot không token.")
    if not initial_qwen_candidates:
        qwen_download_path = MODEL_DOWNLOAD_DIR / "Qwen3-8B"
        print(f"Downloading {QWEN_REPO_ID} -> {qwen_download_path}")
        snapshot_download(repo_id=QWEN_REPO_ID, local_dir=qwen_download_path, token=hf_token)
    if not initial_e5_candidates:
        e5_download_path = MODEL_DOWNLOAD_DIR / "multilingual-e5-small"
        print(f"Downloading {E5_REPO_ID} -> {e5_download_path}")
        snapshot_download(repo_id=E5_REPO_ID, local_dir=e5_download_path, token=hf_token)
    config_paths = _all_model_configs()
    hf_token = None  # Không giữ token trong state sau khi provision model.

qwen_override = _override(QWEN_MODEL_DIR_OVERRIDE)
if qwen_override:
    QWEN_MODEL_DIR = qwen_override
else:
    qwen_candidates = _qwen_candidates(config_paths)
    QWEN_MODEL_DIR = _ranked_choice(
        qwen_candidates,
        "Qwen3-8B model",
        lambda path: 20 if "8b" in str(path).casefold() else 0,
    )

e5_override = _override(E5_MODEL_DIR_OVERRIDE)
if e5_override:
    E5_MODEL_DIR = e5_override
else:
    e5_candidates = _e5_candidates(config_paths)
    E5_MODEL_DIR = _ranked_choice(
        e5_candidates,
        "multilingual-e5-small model",
        lambda path: 10 if "e5-small" in str(path).casefold() else 0,
    )

supported_kb_suffixes = {".tsv", ".csv", ".jsonl"}
TERMINOLOGY_DIR.mkdir(parents=True, exist_ok=True)

def _discover_kb_files():
    roots = (KAGGLE_INPUT, TERMINOLOGY_DIR)
    return [
        path for root in roots for path in root.rglob("*")
        if path.is_file() and path.suffix.casefold() in supported_kb_suffixes
    ]

def _download_file(url: str, destination: Path) -> str:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    request = urllib.request.Request(url, headers={"User-Agent": "Viettel-AI-Race/1.0"})
    print(f"Downloading terminology: {url}")
    with urllib.request.urlopen(request, timeout=180) as response, temporary.open("wb") as output:
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            output.write(chunk)
    temporary.replace(destination)
    digest = hashlib.md5(destination.read_bytes()).hexdigest()
    print(f"Downloaded {destination.name}: {destination.stat().st_size/1024**2:.2f} MiB, md5={digest}")
    return digest

def _format_icd_code(raw_code: str) -> str:
    code = raw_code.strip().upper().replace(".", "")
    return code if len(code) <= 3 else f"{code[:3]}.{code[3:]}"

def _build_icd_tsv(archive_path: Path, output_path: Path) -> int:
    codes = {}
    with zipfile.ZipFile(archive_path, "r") as archive:
        text_members = [name for name in archive.namelist() if name.casefold().endswith(".txt")]
        if not text_members:
            raise RuntimeError("CDC ICD archive không chứa file .txt")
        for member in text_members:
            raw = archive.read(member)
            text = raw.decode("utf-8-sig", errors="replace")
            for line in text.splitlines():
                parts = line.strip().split(maxsplit=1)
                if len(parts) != 2 or not re.fullmatch(r"[A-Za-z][0-9A-Za-z]{2,6}", parts[0]):
                    continue
                code = _format_icd_code(parts[0])
                label = " ".join(parts[1].split())
                if label:
                    codes.setdefault(code, label)
    if len(codes) < 50000:
        raise RuntimeError(f"CDC ICD parse chỉ có {len(codes):,} mã; từ chối dùng KB không đầy đủ")
    with output_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle, delimiter="\t", lineterminator="\n")
        writer.writerow(["code", "label", "aliases"])
        for code in sorted(codes):
            writer.writerow([code, codes[code], ""])
    return len(codes)

def _build_rxnorm_tsv(archive_path: Path, output_path: Path) -> int:
    names = defaultdict(set)
    preferred = {}
    tty_priority = {"SBD": 0, "SCD": 1, "GPCK": 2, "BPCK": 3, "IN": 4, "PIN": 5, "MIN": 6}
    with zipfile.ZipFile(archive_path, "r") as archive:
        members = [name for name in archive.namelist() if name.upper().endswith("RXNCONSO.RRF")]
        if len(members) != 1:
            raise RuntimeError(f"RxNorm archive phải có đúng một RXNCONSO.RRF, tìm thấy {members}")
        with archive.open(members[0], "r") as binary:
            for raw_line in io.TextIOWrapper(binary, encoding="utf-8", errors="replace"):
                fields = raw_line.rstrip("\r\n").split("|")
                if len(fields) < 17:
                    continue
                rxcui, language, is_preferred = fields[0], fields[1], fields[6]
                source, tty, term, suppress = fields[11], fields[12], fields[14], fields[16]
                if not rxcui.isdigit() or language != "ENG" or source != "RXNORM" or suppress != "N" or not term:
                    continue
                clean_term = " ".join(term.replace("|", " ").split())
                names[rxcui].add(clean_term)
                rank = (0 if is_preferred == "Y" else 1, tty_priority.get(tty, 99), len(clean_term), clean_term.casefold())
                if rxcui not in preferred or rank < preferred[rxcui][0]:
                    preferred[rxcui] = (rank, clean_term)
    if len(names) < 20000:
        raise RuntimeError(f"RxNorm parse chỉ có {len(names):,} RxCUI; từ chối dùng KB không đầy đủ")
    with output_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle, delimiter="\t", lineterminator="\n")
        writer.writerow(["code", "label", "aliases"])
        for rxcui in sorted(names, key=int):
            label = preferred[rxcui][1]
            aliases = sorted((name for name in names[rxcui] if name != label), key=lambda value: (len(value), value.casefold()))
            writer.writerow([rxcui, label, "|".join(aliases[:24])])
    return len(names)

def _provision_official_terminology():
    icd_output = TERMINOLOGY_DIR / "icd10.tsv"
    rx_output = TERMINOLOGY_DIR / "rxnorm.tsv"
    source_report = {}
    if not icd_output.is_file():
        icd_archive = TERMINOLOGY_DIR / "icd10cm_2026.zip"
        icd_md5 = _download_file(ICD_ARCHIVE_URL, icd_archive) if not icd_archive.is_file() else hashlib.md5(icd_archive.read_bytes()).hexdigest()
        if icd_md5 != ICD_ARCHIVE_MD5:
            icd_archive.unlink(missing_ok=True)
            icd_md5 = _download_file(ICD_ARCHIVE_URL, icd_archive)
        if icd_md5 != ICD_ARCHIVE_MD5:
            raise RuntimeError(f"ICD MD5 mismatch: expected {ICD_ARCHIVE_MD5}, got {icd_md5}")
        icd_count = _build_icd_tsv(icd_archive, icd_output)
        source_report["icd"] = {"url": ICD_ARCHIVE_URL, "archive_md5": icd_md5, "entries": icd_count}
        print(f"Built {icd_output}: {icd_count:,} ICD codes")
    if not rx_output.is_file():
        rx_archive = TERMINOLOGY_DIR / "RxNorm_full_prescribe_07062026.zip"
        rx_md5 = _download_file(RXNORM_ARCHIVE_URL, rx_archive) if not rx_archive.is_file() else hashlib.md5(rx_archive.read_bytes()).hexdigest()
        if rx_md5 != RXNORM_ARCHIVE_MD5:
            rx_archive.unlink(missing_ok=True)
            rx_md5 = _download_file(RXNORM_ARCHIVE_URL, rx_archive)
        if rx_md5 != RXNORM_ARCHIVE_MD5:
            raise RuntimeError(f"RxNorm MD5 mismatch: expected {RXNORM_ARCHIVE_MD5}, got {rx_md5}")
        rx_count = _build_rxnorm_tsv(rx_archive, rx_output)
        source_report["rxnorm"] = {"url": RXNORM_ARCHIVE_URL, "archive_md5": rx_md5, "entries": rx_count}
        print(f"Built {rx_output}: {rx_count:,} RxCUIs")
    if source_report:
        (TERMINOLOGY_DIR / "sources.json").write_text(json.dumps(source_report, indent=2) + "\n", encoding="utf-8")

all_kb_files = _discover_kb_files()
has_icd = any("icd" in path.name.casefold() for path in all_kb_files)
has_rxnorm = any(re.search(r"rxnorm|rxcui", path.name, re.IGNORECASE) for path in all_kb_files)

def _apply_vn_alias_patches(icd_path, rxnorm_path):
    """Append Vietnamese aliases from project data/terminology patch files."""
    patch_base = PROJECT_ROOT / "data" / "terminology"
    for kb_path, patch_name in [
        (icd_path, "vn_icd10_aliases.tsv"),
        (rxnorm_path, "vn_rxnorm_aliases.tsv"),
    ]:
        patch_file = patch_base / patch_name
        if not patch_file.is_file() or not kb_path.is_file():
            continue
        patches = {}
        with patch_file.open(encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f, delimiter="	")
            for row in reader:
                code = str(row.get("code") or "").strip()
                raw = str(row.get("aliases") or "").strip()
                if code and raw:
                    patches.setdefault(code, []).extend(
                        a.strip() for a in raw.split("|") if a.strip()
                    )
        if not patches:
            continue
        rows = []
        with kb_path.open(encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f, delimiter="	")
            fieldnames = list(reader.fieldnames or [])
            if "aliases" not in fieldnames:
                fieldnames.append("aliases")
            for row in reader:
                rows.append(dict(row))
        patched = 0
        for row in rows:
            extra = patches.get(row.get("code", ""))
            if not extra:
                continue
            existing = [a.strip() for a in (row.get("aliases") or "").split("|") if a.strip()]
            row["aliases"] = "|".join(dict.fromkeys(existing + extra))
            patched += 1
        tmp = kb_path.with_suffix(kb_path.suffix + ".patching")
        with tmp.open("w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter="	",
                                    extrasaction="ignore", lineterminator="
")
            writer.writeheader()
            writer.writerows(rows)
        tmp.replace(kb_path)
        print(f"VN alias patch: {patched} codes patched in {kb_path.name}")

if AUTO_PROVISION_TERMINOLOGY and (not has_icd or not has_rxnorm):
    _provision_official_terminology()
    all_kb_files = _discover_kb_files()

# Apply Vietnamese alias patches to whichever KB files exist in TERMINOLOGY_DIR.
_apply_vn_alias_patches(TERMINOLOGY_DIR / "icd10.tsv", TERMINOLOGY_DIR / "rxnorm.tsv")

icd_override = _override(ICD_KB_PATH_OVERRIDE)
ICD_KB_PATH = icd_override or _ranked_choice(
    [path for path in all_kb_files if "icd" in path.name.casefold()],
    "ICD terminology",
    lambda path: (100 if str(path).startswith("/kaggle/input") else 0) + (20 if "icd10" in path.name.casefold() else 0) + (5 if path.suffix == ".tsv" else 0),
)

rx_override = _override(RXNORM_KB_PATH_OVERRIDE)
RXNORM_KB_PATH = rx_override or _ranked_choice(
    [path for path in all_kb_files if re.search(r"rxnorm|rxcui", path.name, re.IGNORECASE)],
    "RxNorm terminology",
    lambda path: (100 if str(path).startswith("/kaggle/input") else 0) + (20 if "rxnorm" in path.name.casefold() else 0) + (5 if path.suffix == ".tsv" else 0),
)

required_paths = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "INPUT_DIR": INPUT_DIR,
    "QWEN_MODEL_DIR": QWEN_MODEL_DIR,
    "E5_MODEL_DIR": E5_MODEL_DIR,
    "ICD_KB_PATH": ICD_KB_PATH,
    "RXNORM_KB_PATH": RXNORM_KB_PATH,
}
for name, path in required_paths.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} không tồn tại: {path}")

actual_ids = {path.stem for path in INPUT_DIR.glob("*.txt") if path.stem.isdigit()}
expected_ids = {str(index) for index in range(1, 101)}
if actual_ids != expected_ids:
    raise RuntimeError(
        f"Input phải có đúng 1.txt..100.txt; thiếu={sorted(expected_ids-actual_ids)}, "
        f"thừa={sorted(actual_ids-expected_ids)}"
    )
print("Đã tìm thấy đầy đủ 100 input và tất cả tài nguyên bắt buộc.")

## 3. Cài dependency và khóa chế độ offline

PyPI chỉ được dùng để cài thư viện nếu môi trường Kaggle còn thiếu. Trên P100 (`sm_60`), cell tự cài wheel PyTorch 2.10 CUDA 12.6 rồi chạy phép tính CUDA kiểm chứng; cần Restart Session trước khi chạy notebook mới nếu kernel đã import Torch. Model có thể được tải một lần ở cell trước bằng token trong Kaggle Secrets. Sau cell này, Hugging Face/Transformers bị ép offline và toàn bộ inference dùng weights local.

In [ ]:
import importlib
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version as package_version

if "torch" in sys.modules:
    raise RuntimeError(
        "Torch đã được import trong kernel hiện tại. Hãy Restart Session rồi Run All để có thể đổi wheel CUDA an toàn."
    )

dependency_specs = [
    "huggingface-hub>=0.30,<2",
    "pydantic>=2.10,<3",
    "transformers>=4.51,<6",
    "accelerate>=1.2,<2",
    "bitsandbytes>=0.45,<1",
    "sentence-transformers>=3.3,<6",
]

if INSTALL_DEPENDENCIES:
    print("Đang kiểm tra/cài dependency. Cell có thể mất vài phút...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *dependency_specs],
        check=True,
    )
    gpu_probe = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    print("GPU probe:", gpu_probe)
    try:
        installed_torch = package_version("torch")
    except PackageNotFoundError:
        installed_torch = ""
    if ("P100" in gpu_probe or re.search(r"(?:^|,)\s*6\.0\s*$", gpu_probe)) and "+cu126" not in installed_torch:
        print("P100/sm_60 detected: installing the official PyTorch 2.10 CUDA 12.6 wheel...")
        subprocess.run(
            [
                sys.executable, "-m", "pip", "install", "-q", "--force-reinstall",
                "torch==2.10.0", "--index-url", "https://download.pytorch.org/whl/cu126",
            ],
            check=True,
        )

# Xóa key khỏi process và ép toàn bộ model loader chỉ đọc local.
os.environ.pop("OPENAI_API_KEY", None)
for secret_name in HF_SECRET_CANDIDATES:
    os.environ.pop(secret_name, None)
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTHONHASHSEED"] = "17"

source_path = str(PROJECT_ROOT / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)

required_modules = ["pydantic", "torch", "transformers", "accelerate", "bitsandbytes", "sentence_transformers"]
for module_name in required_modules:
    module = importlib.import_module(module_name)
    version = getattr(module, "__version__", "unknown")
    print(f"{module_name}: {version}")

import torch
if torch.cuda.is_available():
    capability = torch.cuda.get_device_capability(0)
    compiled_arches = torch.cuda.get_arch_list()
    print("CUDA capability:", capability, "compiled arches:", compiled_arches)
    if capability == (6, 0) and "sm_60" not in compiled_arches:
        raise RuntimeError("Torch wheel vẫn không chứa sm_60; restart session rồi chạy lại notebook từ đầu.")
    probe_tensor = torch.ones(1, device="cuda") + 1
    assert probe_tensor.item() == 2

import medical_coder
print("medical_coder source:", Path(medical_coder.__file__).resolve())
print("Offline mode enabled; external inference APIs are not used.")

## 4. Kiểm tra GPU, model, terminology và ngân sách 9B

In [ ]:
import csv
import shutil
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Chưa bật GPU: Kaggle Notebook → Settings → Accelerator → GPU")

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"GPU: {gpu_name} ({gpu_memory_gb:.1f} GiB)")
if gpu_memory_gb < 14 and QUANTIZATION == "4bit":
    print("Cảnh báo: VRAM thấp hơn 14 GiB; theo dõi OOM ở smoke test.")

manifest_path = PROJECT_ROOT / "config" / "model_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
calculated_total = sum(item["declared_parameters"] for item in manifest["models"])
if calculated_total != manifest["declared_total_parameters"]:
    raise RuntimeError("model_manifest.json có tổng tham số không nhất quán")
if calculated_total > manifest["parameter_limit"]:
    raise RuntimeError("Tổng weights vượt giới hạn 9B")
print(f"Model budget: {calculated_total/1e9:.3f}B / {manifest['parameter_limit']/1e9:.1f}B")
print(f"Headroom: {manifest['declared_headroom']/1e6:.0f}M parameters")

qwen_config = _load_config(QWEN_MODEL_DIR / "config.json")
if qwen_config.get("model_type") != "qwen3":
    raise RuntimeError(f"Không phải Qwen3 checkpoint: {QWEN_MODEL_DIR}")
if int(qwen_config.get("num_hidden_layers", 0)) < 30:
    raise RuntimeError("Checkpoint có vẻ không phải Qwen3-8B")
if not list(QWEN_MODEL_DIR.glob("*.safetensors")) and not (QWEN_MODEL_DIR / "model.safetensors.index.json").is_file():
    raise RuntimeError("Qwen model directory không có weights safetensors")
if not (E5_MODEL_DIR / "config.json").is_file():
    raise RuntimeError("E5 model directory thiếu config.json")

from medical_coder.terminology import load_terminology
icd_entries = load_terminology(ICD_KB_PATH)
rxnorm_entries = load_terminology(RXNORM_KB_PATH)
print(f"ICD entries: {len(icd_entries):,}")
print(f"RxNorm entries: {len(rxnorm_entries):,}")
if len(icd_entries) < 1000:
    print("Cảnh báo: ICD KB có ít hơn 1.000 mã; coverage có thể quá thấp.")
if len(rxnorm_entries) < 1000:
    print("Cảnh báo: RxNorm KB có ít hơn 1.000 mã; coverage có thể quá thấp.")

disk = shutil.disk_usage(WORKING_DIR)
print(f"Working disk free: {disk.free/(1024**3):.1f} GiB")
print("Preflight checks passed.")

## 5. Chuẩn bị lệnh inference

Output được tạo trong `/kaggle/working`, không ghi vào Dataset read-only. Cache được giữ để có thể resume khi session gián đoạn.

In [ ]:
import hashlib
import time

if RESET_OUTPUT_BEFORE_RUN:
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

subprocess_env = os.environ.copy()
subprocess_env["PYTHONPATH"] = str(PROJECT_ROOT / "src")

def base_predict_command():
    embedding_model = str(E5_MODEL_DIR) if USE_SEMANTIC_RETRIEVAL else "none"
    return [
        sys.executable, "-m", "medical_coder", "predict",
        "--input-dir", str(INPUT_DIR),
        "--output-dir", str(OUTPUT_DIR),
        "--cache-dir", str(CACHE_DIR),
        "--model-path", str(QWEN_MODEL_DIR),
        "--embedding-model", embedding_model,
        "--embedding-device", EMBEDDING_DEVICE,
        "--icd-kb", str(ICD_KB_PATH),
        "--rxnorm-kb", str(RXNORM_KB_PATH),
        "--quantization", QUANTIZATION,
        "--workers", "1",
        "--max-candidates", str(MAX_CANDIDATES),
        "--retrieval-top-k", str(RETRIEVAL_TOP_K),
        "--max-input-tokens", str(MAX_INPUT_TOKENS),
        "--max-new-tokens", str(MAX_NEW_TOKENS),
    ]

def run_streaming(command, log_path: Path):
    print("Running:", " ".join(command))
    started = time.time()
    with log_path.open("a", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=subprocess_env,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log_handle.write(line)
            log_handle.flush()
        return_code = process.wait()
    elapsed = time.time() - started
    if return_code != 0:
        raise RuntimeError(f"Command failed with exit code {return_code}; xem {log_path}")
    print(f"Completed in {elapsed/60:.1f} minutes")
    return elapsed

LOG_PATH.write_text("", encoding="utf-8")
print("Inference command prepared.")

## 6. Smoke test

Hai record đầu được chạy trước. Nếu cell này lỗi OOM hoặc lỗi quantization, dừng và xử lý trước khi chạy toàn bộ.

In [ ]:
smoke_elapsed = 0.0
if RUN_SMOKE_TEST:
    smoke_command = base_predict_command() + [
        "--ids", SMOKE_IDS,
        "--overwrite",
        "--no-zip",
    ]
    smoke_elapsed = run_streaming(smoke_command, LOG_PATH)
    smoke_ids = [str(int(value.strip())) for value in SMOKE_IDS.split(",")]
    for record_id in smoke_ids:
        output_path = OUTPUT_DIR / f"{record_id}.json"
        if not output_path.is_file():
            raise RuntimeError(f"Smoke test thiếu {output_path}")
        parsed = json.loads(output_path.read_text(encoding="utf-8"))
        print(f"Smoke {record_id}: {len(parsed)} entities")
    print("Smoke test passed.")
else:
    print("Smoke test disabled by configuration.")

## 7. Chạy đủ 100 record và tạo output.zip

Các response hợp lệ của smoke test được lấy từ cache, nên hai record đầu không phải suy luận lại.

In [ ]:
full_command = base_predict_command() + [
    "--zip-path", str(ZIP_PATH),
]
full_elapsed = run_streaming(full_command, LOG_PATH)
if not ZIP_PATH.is_file():
    raise RuntimeError("Inference hoàn tất nhưng chưa tạo output.zip")
print(f"Created: {ZIP_PATH} ({ZIP_PATH.stat().st_size/(1024**2):.2f} MiB)")

## 8. Validation cuối và kiểm tra ZIP độc lập

In [ ]:
import zipfile

validate_command = [
    sys.executable, "-m", "medical_coder", "validate",
    "--input-dir", str(INPUT_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--zip-path", str(ZIP_PATH),
]
run_streaming(validate_command, LOG_PATH)

expected_members = [f"output/{index}.json" for index in range(1, 101)]
with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    members = archive.namelist()
    if members != expected_members:
        missing = sorted(set(expected_members) - set(members))
        extra = sorted(set(members) - set(expected_members))
        raise RuntimeError(f"ZIP sai member/order; missing={missing}, extra={extra}")
    corrupt = archive.testzip()
    if corrupt is not None:
        raise RuntimeError(f"ZIP corrupt tại member: {corrupt}")
    for record_id in ("1", "50", "100"):
        entities = json.loads(archive.read(f"output/{record_id}.json"))
        if not isinstance(entities, list):
            raise RuntimeError(f"output/{record_id}.json không phải JSON list")

output_files = sorted(
    (path for path in OUTPUT_DIR.glob("*.json") if path.stem.isdigit()),
    key=lambda path: int(path.stem),
)
entity_counts = [len(json.loads(path.read_text(encoding="utf-8"))) for path in output_files]
zip_sha256 = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
run_report = {
    "status": "validated",
    "gpu": gpu_name,
    "gpu_memory_gib": round(gpu_memory_gb, 2),
    "model": str(QWEN_MODEL_DIR),
    "embedding_model": str(E5_MODEL_DIR) if USE_SEMANTIC_RETRIEVAL else None,
    "declared_total_parameters": calculated_total,
    "input_records": 100,
    "output_records": len(output_files),
    "total_entities": sum(entity_counts),
    "min_entities_per_record": min(entity_counts),
    "max_entities_per_record": max(entity_counts),
    "smoke_seconds": round(smoke_elapsed, 2),
    "full_run_seconds": round(full_elapsed, 2),
    "output_zip": str(ZIP_PATH),
    "output_zip_bytes": ZIP_PATH.stat().st_size,
    "output_zip_sha256": zip_sha256,
}
RUN_REPORT_PATH.write_text(
    json.dumps(run_report, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print(json.dumps(run_report, ensure_ascii=False, indent=2))
print("FINAL VALIDATION PASSED")

## 9. Tải kết quả

`output.zip` là file nộp cho Ban Tổ chức. `run_report.json` và `inference.log` chỉ phục vụ kiểm tra, không nằm trong ZIP.

In [ ]:
from IPython.display import FileLink, display

display(FileLink(str(ZIP_PATH), result_html_prefix="Tải file nộp: "))
display(FileLink(str(RUN_REPORT_PATH), result_html_prefix="Tải báo cáo chạy: "))
display(FileLink(str(LOG_PATH), result_html_prefix="Tải inference log: "))
print("Nếu liên kết không tải trực tiếp, mở panel Files của Kaggle và tải /kaggle/working/output.zip")